# Multivariate  Decoding Analysis Pipeline

---

**Platforms and Packages Used:**
* **Python Version:** 3.11.8 (Anaconda 24.3.0)
* **NumPy Version:** 1.26.4
* **Pandas Version:** 2.3.2
* **SciPy Version:** 1.13.1
* **Pingouin Version:** 0.5.5
* **Scikit-learn Version:** 1.4.1
* **MNE-Python Version:** 1.10.0

**Terminology Mapping (Code vs. Manuscript):**

| Code / Filename | Manuscript Terminology |
| :--- | :--- |
| `"subject"` | "participant" |
| `"decisions"` | "action conditions" |
| `"PreAssn"` | "Pre-association phase" |
| `"Sess1"` | "Post-association Session1" |

| `"Sess2"` | "Post-association Session2" |

**Overview:**

This notebook performs time-resolved multivariate decoding on Current Source Density (CSD) data to investigate the generalized neural representations of action-effect prediction during the pre-action period. 

**Pipeline Steps:**
1. **Environmental Setup & Parameters:** Load required packages, define working directories, set up experimental parameters, and configure decoding parameters.
2. **Cross-Validation Generation:** Generate balanced cross-validation indices (20 repeats × 8 folds) to ensure reproducibility across subjects and models.
3. **Model Training & Testing:** Implement a sliding-window temporal decoding pipeline.
4. **Performance Aggregation:** Compute and compile performance matrices (ROC-AUC and Accuracy) across all CV iterations.
5. **Permutation Testing for AUC:** Establish the chance-level null distributions by executing the identical pipeline with randomly shuffled condition labels.
6. **Predictive Power Indix Calculation (PPI):** Extract model weights and transform them into spatial activation patterns (i.e., PPI), specifically applying Log-transformed Normalization for topographical interpretation.

## Environmental Setup & Parameters

In [1]:
import os
import sys
import mne
import time
import numpy as np
import pandas as pd
from pathlib import Path
import pingouin as pg
from scipy import stats
from scipy.stats import ttest_1samp
from itertools import product
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold
from skimage.measure import label, regionprops

# Set standard output formats
np.set_printoptions(precision=3, linewidth=120)

# Define paths
# root_dir = Path('/path/to/your/project/directory') # Update with your actual path
# data_dir = root_dir / 'PipelineData/csddata_decode'
# stat_dir = root_dir / 'statdata'
# mat_dir  = root_dir / 'matdata'
# decode_dir = root_dir / 'decodedata'
# cv_idx_dir = decode_dir / 'CV_idx_files'
# perf_dir   = decode_dir / 'Perform_matrix'
# for d in [cv_idx_dir, perf_dir]: d.mkdir(parents=True, exist_ok=True)
root_dir = Path('/home/zhengting/Insync/OneDrive/ParisCite/1_MultivariateProcedure/')
data_dir = Path('/home/zhengting/BackupData/ParisCite/1_MultivariateProcedure/PipelineData_3rd/decode_new_noout_interbdchs_CSD/')
stat_dir = root_dir / 'code/Github_pub/statdata'
mat_dir  = root_dir / 'code/Github_pub/matdata'
decode_dir = Path('/home/zhengting/BackupData/ParisCite/1_MultivariateProcedure/PipelineData_3rd/decode_data_py')
# cv_idx_dir = decode_dir / 'CV_idx_files_update_final'
cv_idx_dir = root_dir / 'code/Github_pub/decodedata/CV_idx_files'
# perf_dir   = decode_dir / 'MatData_final'
perf_dir   = root_dir / 'code/Github_pub/decodedata/PerfData_final'
for d in [cv_idx_dir, perf_dir]: d.mkdir(parents=True, exist_ok=True)

# Experimental factors
eegtype = 'CSD'
period = 'PreAction'
# day_names = ['Day1', 'Day2']
decis_names = ['Cue', 'Vol']
stim_names = ['Vis', 'Aud']
sess_names = ['Sess1', 'Sess2']
decisions = ['Cued', 'Voluntary']
stimuli = ['Visual', 'Auditory']
sessions = ['Session1', 'Session2']


# Strict Participant Filtering Database (Trials >= 40 inclusion requirement)
# and details see in Table S1 and AEP_02 (cell of 'Sample Selection')
num_subs = 20
dict_model_sids = {
    'Cued_Visual': np.array([1, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 15, 16, 17, 18, 20]),
    'Cued_Auditory': np.array([1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 18, 19, 20]),
    'Voluntary_Visual': np.array([1, 5, 6, 9, 11, 13, 14, 16, 17, 20]),
    'Voluntary_Auditory': np.array([1, 4, 6, 9, 10, 13, 14, 16, 17])
}

# Parameters
n_repeat = 20
n_splits = 8
n_perm = 1000
random_seed = 100

# Load metadata (Channels & Times) from a template file
# template_path = data_dir / '1/s1_Day1_CuedAuditory_Dsampled_CSD.set'
template_path = data_dir / '1/s1_Session1_CuedAuditory_CSD_Decode.set'
data_raw_template = mne.io.read_epochs_eeglab(template_path, verbose='CRITICAL')
ch_names = np.asarray(data_raw_template.info['ch_names'])
times = data_raw_template.times
times_ms = (times * 1000).astype(int)
time_start_idx = 35
time_zero_idx  = 95
times_ms_pre = times_ms[time_start_idx:time_zero_idx+1]


## Cross-Validation Generation

In [78]:
print("Initializing Balance Validation Constraints...")

random_seed = 100
np.random.seed(random_seed)
random_ints = np.random.randint(0, 300, size=n_repeat)
print('np_random_seed:', random_ints)

## load counts of each trial type for each participants, the file generated in AEP_02
df_behavior_count = pd.read_excel(stat_dir/'Behavioral_subjects.xlsx', sheet_name='Count')
df_behavior_count.index = np.arange(1, num_subs+1)

# for decis, stim, day in product(decis_names, stim_names, day_names):
#     decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
#     stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
#     paraname = f'{decision}_{stimulus}'
#     trialinfo = f'{decis}_{day}_No{stim}{period}'
for decis, stim, sess in product(decis_names, stim_names, sess_names):
    decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
    stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
    session = 'Session1' if sess == 'Sess1' else 'Session2'
    paraname = f'{decision}_{stimulus}'
    trialinfo = f'{decis}_{sess}_No{stim}{period}'
    
    if paraname not in dict_model_sids: 
        continue
        
    for id_rand, rand_state in enumerate(random_ints):
        for sid in dict_model_sids[paraname]:
            fname_train = cv_idx_dir / f'{n_splits}Folds_{trialinfo}_S{sid}_train_Round{id_rand:02d}_Seed{rand_state}.xlsx'
            fname_test  = cv_idx_dir / f'{n_splits}Folds_{trialinfo}_S{sid}_test_Round{id_rand:02d}_Seed{rand_state}.xlsx'
            
            if fname_train.exists() and fname_test.exists():
                continue # Skip file building if already compiled
                
            # --- Dynamic Parity Selection Loop ---
            # len_pre = subs_rt[f'Day1_{decision}No{stimulus}StimPre'][sid]
            # len_Post = subs_rt[f'{day}_{decision}{stimulus}'][sid]
            len_pre = df_behavior_count[f'Session1_{decision}No{stimulus}StimPre'][sid]
            # len_post = df_behavior_count[f'{session}_{decision}{stimulus}'][sid]
            len_post = df_behavior_count[f'{session}_{decision}{stimulus}'][sid]+\
                       df_behavior_count[f'{session}_{decision}No{stimulus}StimPost'][sid]
            
            df_pre = pd.DataFrame({'GroundTruth': [0] * len_pre, 'TID': [f'Pre_{i:03d}' for i in range(len_pre)]}).set_index('TID')
            df_post = pd.DataFrame({'GroundTruth': [1] * len_post, 'TID': [f'Post_{i:03d}' for i in range(len_post)]}).set_index('TID')
            
            # Subsample post-stimulus trials to achieve absolute 1:1 balance mapping
            df_post_balanced = df_post.sample(frac=1, random_state=rand_state)[:len_pre]
            df_combined = pd.concat([df_pre, df_post_balanced])
            
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rand_state)
            
            with pd.ExcelWriter(fname_train) as train_writer, pd.ExcelWriter(fname_test) as test_writer:
                for kidx, (train_idx, test_idx) in enumerate(skf.split(df_combined, df_combined['GroundTruth'])):
                    df_train = df_combined.iloc[train_idx].copy()
                    df_test  = df_combined.iloc[test_idx].copy()
                    df_train['idx'] = train_idx
                    df_test['idx']  = test_idx
                    df_train.to_excel(train_writer, sheet_name=f'Fold{kidx+1}')
                    df_test.to_excel(test_writer, sheet_name=f'Fold{kidx+1}')

print("All Stratified CV Sheets generated and stored successfully.")

Initializing Balance Validation Constraints...
np_random_seed: [  8 280  79  53  66 226  14 290 240 280 143 228  58 137  93  86 155 141 245 211]
All Stratified CV Sheets generated and stored successfully.


## Model Training & Testing
Parallelization Pipeline via Multiprocessing Pools

In [131]:
print("Packaging Task Payload for Parallel Workers...")
from AEP_04a_decoding_worker_pipeline import run_subject_decoding_pipeline

task_payload = []
# for decis, stim, day in product(decis_names, stim_names, day_names):
#     decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
#     stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
#     paraname = f'{decision}_{stimulus}'
for decis, stim, sess in product(decis_names[1:], stim_names[1:], sess_names[1:]):
    decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
    stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
    paraname = f'{decision}_{stimulus}'
    
    if paraname not in dict_model_sids:
        continue
        
    for sid in dict_model_sids[paraname]:
        # Pack target combinations with shared random seeds AND directory paths
        # We convert paths to str() to ensure safe multiprocessing serialization
        task_payload.append((
            decis, 
            stim, 
            sess, # day
            sid, 
            random_ints, 
            str(cv_idx_dir), 
            str(perf_dir), 
            str(data_dir)
        ))

print(f"Total tasks compiled into execution queue: {len(task_payload)}")

# Multi-Core Processing Allocation Gate
if __name__ == '__main__':
    print(f"Starting parallel execution for {len(task_payload)} tasks using 16 cores...")
    with Pool(processes=16) as pool:
        results = pool.map(run_subject_decoding_pipeline, task_payload)
    
    for res in results: print(res)        
    print("\nParallel Processing Optimization Stage Concluded.")

Packaging Task Payload for Parallel Workers...
Total tasks compiled into execution queue: 9
Starting parallel execution for 9 tasks using 16 cores...
Subject 1 | Voluntary Auditory complete.
Subject 4 | Voluntary Auditory complete.
Subject 6 | Voluntary Auditory complete.
Subject 9 | Voluntary Auditory complete.
Subject 10 | Voluntary Auditory complete.
Subject 13 | Voluntary Auditory complete.
Subject 14 | Voluntary Auditory complete.
Subject 16 | Voluntary Auditory complete.
Subject 17 | Voluntary Auditory complete.

Parallel Processing Optimization Stage Concluded.


## Performance Aggregation

In [114]:
def compute_vectorized_auc_matrix(prob_matrix_cv, y_true_vector):
    """
    Computes rapid non-parametric ROC-AUC using broadcast matrix scaling.
    Shape Input: prob_matrix_cv -> (n_repeat, n_trials, n_times_train, n_times_general)
    """
    n_repeat, n_trials, n_times_train, n_times_general = prob_matrix_cv.shape
    prob_flat = np.transpose(prob_matrix_cv, (0, 2, 3, 1)).reshape(-1, n_trials)
    
    y_expanded = y_true_vector[np.newaxis, np.newaxis, np.newaxis, :]
    y_broadcast = np.broadcast_to(y_expanded, (n_repeat, n_times_train, n_times_general, n_trials)).reshape(-1, n_trials)
    
    ranks = np.argsort(np.argsort(prob_flat, axis=1), axis=1) + 1
    n_pos = np.sum(y_broadcast == 1, axis=1)
    n_neg = n_trials - n_pos
    
    sum_ranks_pos = np.sum(ranks * (y_broadcast == 1), axis=1)
    auc_flat = (sum_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    
    return auc_flat.reshape(n_repeat, n_times_train, n_times_general)

In [133]:
print("Aggregating Performance Matrices from Worker Outputs...")

# Initialize dictionaries to store the final metrics across subjects
dict_auc_results = {}
dict_acc_results = {}

# for decis, stim, day in product(decis_names, stim_names, day_names):
#     decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
#     stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
#     paraname = f'{decision}_{stimulus}'
#     paraname2 = f'{decision}_{stimulus}_{day}'
#     trialinfo = f'{decis}_{day}_No{stim}{period}'
for decis, stim, sess in product(decis_names[1:], stim_names[1:], sess_names[1:]):
    decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
    stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
    session = 'Session1' if sess == 'Sess1' else 'Session2'
    paraname = f'{decision}_{stimulus}'
    paraname2 = f'{decision}_{stimulus}_{session}'
    trialinfo = f'{decis}_{sess}_No{stim}{period}'

    # Skip if condition is not in our inclusion dictionary
    if paraname not in dict_model_sids:
        continue

    # Initialize sub-dictionaries for the current condition
    if paraname2 not in dict_auc_results:
        dict_auc_results[paraname2] = {}
        dict_acc_results[paraname2] = {}

    for sid in dict_model_sids[paraname]:
        sid_dir = perf_dir / str(sid)
        savefile_template = f'{n_repeat}Rounds_{n_splits}Folds_ElasticNet_{trialinfo}_S{sid}'
        # savefile_template = f'{n_repeat}Rounds_{n_splits}Folds_ElasticNetStrict_{trialinfo}_S{sid}'

        # 1. Load probability and prediction matrices generated by the worker
        try:
            with np.load(sid_dir / f'{savefile_template}_prob_matrix_cv.npz') as data_prob:
                prob_matrix_cv = data_prob['arr_0']
            with np.load(sid_dir / f'{savefile_template}_pred_matrix_cv.npz') as data_pred:
                pred_matrix_cv = data_pred['arr_0']
        except FileNotFoundError:
            print(f"  [Warning] Missing result files for S{sid} ({paraname2}). Skipping...")
            continue
        except Exception as e:
            print(f"  [Error] Failed to load data for S{sid} ({paraname2}): {e}")
            continue

        # 2. Reconstruct y_true_vector (Ground Truth Labels)
        # Since stratification maintains the same global index across rounds, we only need to load Round 00 to extract the true labels sequence.
        id_rand, rand_state = 0, random_ints[0]
        fname_train = cv_idx_dir / f'{n_splits}Folds_{trialinfo}_S{sid}_train_Round{id_rand:02d}_Seed{rand_state}.xlsx'
        fname_test  = cv_idx_dir / f'{n_splits}Folds_{trialinfo}_S{sid}_test_Round{id_rand:02d}_Seed{rand_state}.xlsx'
        
        df_train = pd.read_excel(fname_train, sheet_name='Fold1', index_col=0)
        df_test  = pd.read_excel(fname_test, sheet_name='Fold1', index_col=0)
        
        # Concatenate and sort by the original global index to match the matrix trial dimension
        y_all = pd.concat([df_train, df_test]).sort_values(by=['idx'])
        y_true_vector = y_all['GroundTruth'].values

        # 3. Calculate Vectorized ROC-AUC Matrix
        # Shape output: (n_repeat, n_times_train, n_times_general)
        auc_matrix = compute_vectorized_auc_matrix(prob_matrix_cv, y_true_vector)

        # 4. Calculate Accuracy (ACC) Matrix via Broadcasted Boolean Matching
        # Expand true labels to shape (1, n_trials, 1, 1) for broadcasting
        y_expanded = y_true_vector[np.newaxis, :, np.newaxis, np.newaxis]
        # Calculate mean across the trial axis (axis=1)
        acc_matrix = (pred_matrix_cv == y_expanded).mean(axis=1)

        # 5. Store in dictionaries
        dict_auc_results[paraname2][sid] = auc_matrix
        dict_acc_results[paraname2][sid] = acc_matrix

        # print(f"  Processed S{sid:02d} | {decision} {stimulus} {day} | AUC shape: {auc_matrix.shape}")
        print(f"  Processed S{sid:02d} | {decision} {stimulus} {session} | AUC shape: {auc_matrix.shape}")

# 6. Save the aggregated master dictionaries for fast loading in plotting scripts
np.savez_compressed(perf_dir / 'Aggregated_AUC_CV.npz', **dict_auc_results)
np.savez_compressed(perf_dir / 'Aggregated_ACC_CV.npz', **dict_acc_results)

print("All performance matrices successfully aggregated and saved.")

Aggregating Performance Matrices from Worker Outputs...
  Processed S01 | Voluntary Auditory Session2 | AUC shape: (20, 96, 96)
  Processed S04 | Voluntary Auditory Session2 | AUC shape: (20, 96, 96)
  Processed S06 | Voluntary Auditory Session2 | AUC shape: (20, 96, 96)
  Processed S09 | Voluntary Auditory Session2 | AUC shape: (20, 96, 96)
  Processed S10 | Voluntary Auditory Session2 | AUC shape: (20, 96, 96)
  Processed S13 | Voluntary Auditory Session2 | AUC shape: (20, 96, 96)
  Processed S14 | Voluntary Auditory Session2 | AUC shape: (20, 96, 96)
  Processed S16 | Voluntary Auditory Session2 | AUC shape: (20, 96, 96)
  Processed S17 | Voluntary Auditory Session2 | AUC shape: (20, 96, 96)
All performance matrices successfully aggregated and saved.


## Permutation Testing for AUC

In [3]:
def compute_auc_perms_np(prob_matrix_cv_np, y_perms_np, batch_size=2):
    """
    Computes permutation ROC-AUC using highly optimized memory-batched numpy broadcasting.
    """
    n_repeat, n_trials, n_times_train, n_times_general = prob_matrix_cv_np.shape
    n_perm = y_perms_np.shape[1]
    
    auc_list = []
    for start in range(0, n_repeat, batch_size):
        T_start = time.time()
        end = min(start + batch_size, n_repeat)
        
        # Expand the shape of the probability matrix to (batch_size, n_perm, n_times_train, n_times_general, n_trials)
        prob_expanded = np.transpose(np.expand_dims(prob_matrix_cv_np[start:end,...], axis=1), (0, 1, 3, 4, 2))
        prob_broadcasted = np.broadcast_to(prob_expanded, (end-start, n_perm, n_times_train, n_times_general, n_trials))
        prob_flat = prob_broadcasted.reshape(-1, n_trials)
        del prob_expanded, prob_broadcasted
        
        # Expand and align the permuted label shape
        y_perms_expanded = np.transpose(y_perms_np[start:end, :, :, np.newaxis, np.newaxis], (0, 1, 3, 4, 2))
        y_perms_broadcasted = np.broadcast_to(y_perms_expanded, (end-start, n_perm, n_times_train, n_times_general, n_trials))
        y_perms_flat = y_perms_broadcasted.reshape(-1, n_trials)
        del y_perms_expanded, y_perms_broadcasted
        
        # Rank-based computation
        ranks = np.argsort(np.argsort(prob_flat, axis=1), axis=1) + 1
        del prob_flat
        
        n_pos = np.sum(y_perms_flat == 1, axis=1)
        n_neg = n_trials - n_pos
        sum_ranks_pos = np.sum(ranks * (y_perms_flat == 1), axis=1)
        del y_perms_flat
        
        auc_batch = (sum_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
        auc_list.append(auc_batch.reshape(end-start, n_perm, n_times_train, n_times_general))
        
        T_end = time.time()
        del ranks, n_pos, n_neg, sum_ranks_pos, auc_batch
        print(f'  {savefile_template}_AUC_BATCH {start}-{end-1}: {(T_end-T_start):.2f}s', file=sys.stdout)
        
    auc_perms_np = np.concatenate(auc_list)
    return auc_perms_np

In [4]:
def cluster_size_null_t_pos(permutedTstat, sigThresh):
    """Extract the size and sumstat of the largest cluster in a single Null T-map"""
    cluster_info = np.zeros(2)
    # POSITIVE-valued clusters (One-tailed)
    labeled = label(permutedTstat > sigThresh)
    clusters = regionprops(labeled)
    if len(clusters) > 0:
        # Find the cluster with the largest area
        No_maxcluster = np.argmax([clu.num_pixels for clu in clusters])
        cluster_info[0] = clusters[No_maxcluster].num_pixels
        cluster_info[1] = np.sum([permutedTstat[i, j] for i, j in clusters[No_maxcluster].coords])
    return cluster_info

def perm_test_Perform_MikeXCOHEN_new(TrueTmap, PermTmap, nSubs, alpha_cluster_forming, pVal, tail=1):
    """
    TrueTmap: (n_time_train, n_times_general)
    PermTmap: (n_perm, n_time_train, n_times_general)
    """
    n_perm = PermTmap.shape[0]
    
    q = (1 - alpha_cluster_forming) * 100 if tail == 1 else (1 - alpha_cluster_forming / 2) * 100
    degrees_of_freedom = nSubs - 1
    sigThresh = stats.t.ppf(q / 100, df=degrees_of_freedom)

    if tail == 1:
        cluster_infos = Parallel(n_jobs=-1)(
            delayed(cluster_size_null_t_pos)(PermTstat, sigThresh) for PermTstat in PermTmap
        )
        cluster_infos = np.stack(cluster_infos)

    labeled = label(TrueTmap > sigThresh)
    regions = regionprops(labeled)

    regions_info = dict()
    regions_info['size'] = [r.num_pixels for r in regions]
    regions_info['sumstats'] = [np.sum([TrueTmap[i, j] for (i, j) in r.coords]) for r in regions]
    regions_info['pval_size'] = [(np.sum(cluster_infos[:, 0] >= s) + 1) / (n_perm + 1) for s in regions_info['size']]
    regions_info['pval_sumstats'] = [(np.sum(cluster_infos[:, 1] >= s) + 1) / (n_perm + 1) for s in regions_info['sumstats']]
    
    regions_info_df = pd.DataFrame(regions_info)
    
    return regions, regions_info_df, cluster_infos, sigThresh

### Permuted AUC matrix

In [155]:
# To prevent memory overflow, first save the permuted AUC matrix.
print("Starting Permutation Testing Evaluation for AUC ...")

batch_size_limit = 2
random_seed = 100
rng = np.random.default_rng(random_seed)

# for decis, stim, day in product(decis_names, stim_names, day_names):
#     decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
#     stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
#     paraname = f'{decision}_{stimulus}'
#     paraname2 = f'{decision}_{stimulus}_{day}'
#     trialinfo = f'{decis}_{day}_No{stim}{period}'
for decis, stim, sess in product(decis_names[1:], stim_names[1:], sess_names[1:]):
    decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
    stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
    session = 'Session1' if sess == 'Sess1' else 'Session2'
    paraname = f'{decision}_{stimulus}'
    paraname2 = f'{decision}_{stimulus}_{session}'
    trialinfo = f'{decis}_{sess}_No{stim}{period}'

    if paraname not in dict_model_sids:
        continue

    for sid in dict_model_sids[paraname]:
        sid_dir = perf_dir / str(sid)
        savefile_template = f'{n_repeat}Rounds_{n_splits}Folds_ElasticNet_{trialinfo}_S{sid}'
        # savefile_template = f'{n_repeat}Rounds_{n_splits}Folds_ElasticNetStrict_{trialinfo}_S{sid}'
        perm_save_path = sid_dir / f'{savefile_template}_auc_perms.npz'
        
        if perm_save_path.exists():
            print(f"  [Skip] S{sid:02d} ({paraname2}) Permutations already computed.")
            continue

        print(f"\n--- Processing Permutations for S{sid:02d} | {paraname2} ---")
        
        try:
            with np.load(sid_dir / f'{savefile_template}_prob_matrix_cv.npz') as data_prob:
                prob_matrix_cv = data_prob['arr_0']
        except FileNotFoundError:
            print(f"  [Warning] Missing probability matrix for S{sid}. Skipping...")
            continue

        id_rand, rand_state = 0, random_ints[0]
        fname_train = cv_idx_dir / f'{n_splits}Folds_{trialinfo}_S{sid}_train_Round{id_rand:02d}_Seed{rand_state}.xlsx'
        fname_test  = cv_idx_dir / f'{n_splits}Folds_{trialinfo}_S{sid}_test_Round{id_rand:02d}_Seed{rand_state}.xlsx'
        
        df_train = pd.read_excel(fname_train, sheet_name='Fold1', index_col=0)
        df_test  = pd.read_excel(fname_test, sheet_name='Fold1', index_col=0)
        y_all = pd.concat([df_train, df_test]).sort_values(by=['idx'])
        y_true_vector = y_all['GroundTruth'].values
        n_trials = len(y_true_vector)
        
        T_perm_start = time.time()
        shuffled_labels = np.stack([rng.permutation(y_true_vector) for _ in range(n_perm * n_repeat)])
        y_perms_np = shuffled_labels.reshape(n_repeat, n_perm, n_trials)
        
        auc_perms_cv = compute_auc_perms_np(prob_matrix_cv, y_perms_np, batch_size=batch_size_limit)
        
        np.savez_compressed(perm_save_path, arr_0=auc_perms_cv)
        T_perm_end = time.time()
        
        del prob_matrix_cv, y_perms_np, auc_perms_cv
        
        print(f"  [Success] Saved Permutations for S{sid:02d}. Total Time: {(T_perm_end - T_perm_start):.2f}s")

print("\nAll Permutation Testing Null Distributions Generated and Saved Locally.")

Starting Permutation Testing Evaluation for AUC ...
  [Skip] S01 (Voluntary_Auditory_Session2) Permutations already computed.
  [Skip] S04 (Voluntary_Auditory_Session2) Permutations already computed.
  [Skip] S06 (Voluntary_Auditory_Session2) Permutations already computed.
  [Skip] S09 (Voluntary_Auditory_Session2) Permutations already computed.
  [Skip] S10 (Voluntary_Auditory_Session2) Permutations already computed.
  [Skip] S13 (Voluntary_Auditory_Session2) Permutations already computed.
  [Skip] S14 (Voluntary_Auditory_Session2) Permutations already computed.
  [Skip] S16 (Voluntary_Auditory_Session2) Permutations already computed.
  [Skip] S17 (Voluntary_Auditory_Session2) Permutations already computed.

All Permutation Testing Null Distributions Generated and Saved Locally.


### Cluster-Based Permutation Statistics

In [5]:
print("Running Cluster-Based Permutation Statistics...")

alpha_cluster_forming = 0.025
pVal = 0.05
chance = 0.5
batch_size = 5000 

# Load saved data
aggregated_auc_path = perf_dir / 'Aggregated_AUC_CV.npz'
dict_auc_results = np.load(aggregated_auc_path, allow_pickle=True)

dict_Perform_PermTest_ResInfos = {}

# for decis, stim, day in product(decis_names, stim_names, day_names):
#     decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
#     stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
#     paraname2 = f'{decision}_{stimulus}_{day}'
#     trialinfo = f'{decis}_{day}_No{stim}{period}'
for decis, stim, sess in product(decis_names[1:], stim_names[1:], sess_names[1:]):
    decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
    stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
    session = 'Session1' if sess == 'Sess1' else 'Session2'
    paraname2 = f'{decision}_{stimulus}_{session}'
    trialinfo = f'{decis}_{sess}_No{stim}{period}'

    if paraname2 not in dict_auc_results:
        continue
    
    print(f"\n--- [Stats] Analyzing: {paraname2} ---")
    T_start = time.time()
    
    # Load subject data
    subject_auc_dict = dict_auc_results[paraname2].item()
    
    # Calculate mean of n_repeat (20 times) for each subject, and stacked as (n_subs, n_times_train, n_times_general)
    matrix_auc_group = np.stack([auc_mat.mean(axis=0) for auc_mat in subject_auc_dict.values()])
    n_subs = matrix_auc_group.shape[0]
    
    # Perform a t-test and the output shape (n_times_train, n_times_general).
    matrix_tmap, _ = ttest_1samp(matrix_auc_group, chance)
    
    # Read and merge the null permutation matrix (PermTmap) of all included subjects.
    perms_cv_group = []
    for sid in subject_auc_dict:
        sid_folder = perf_dir / str(sid)
        savefile_template = f'{n_repeat}Rounds_{n_splits}Folds_ElasticNet_{trialinfo}_S{sid}'
        # savefile_template = f'{n_repeat}Rounds_{n_splits}Folds_ElasticNetStrict_{trialinfo}_S{sid}'
        
        try:
            #  (n_repeat, n_perm, n_times_train, n_times_general) flattened to (n_repeat * n_perm, n_times_train, n_times_general)
            perms_cv = np.load(sid_folder / f'{savefile_template}_auc_perms.npz')['arr_0']
            n_times_train, n_times_general = perms_cv.shape[2], perms_cv.shape[3]
            perms_cv = perms_cv.astype(np.float32).reshape(-1, n_times_train, n_times_general)
            perms_cv_group.append(perms_cv)
        except FileNotFoundError:
            print(f"  [Warning] Missing perm file for S{sid}. Statistics will be compromised!")
            continue
            
    # shape (n_subs, total_perm, n_times_train, n_times_general)
    perms_cv_group = np.stack(perms_cv_group)
    n_perm_all = perms_cv_group.shape[1]
    
    # Batch processing calculation of Permutation T-maps
    print(f"  > Calculating Permutation T-maps for {n_perm_all} permutations...")
    perms_tmap = np.zeros(perms_cv_group.shape[1:], dtype=np.float32)
    
    for start in range(0, n_perm_all, batch_size):
        end = min(start + batch_size, n_perm_all)
        perms_tmap_batch, _ = ttest_1samp(perms_cv_group[:, start:end, ...], chance)
        perms_tmap[start:end, ...] = perms_tmap_batch
    
    # Perform cluster-based permutation test
    print(f"  > Extracting Clusters & Calculating Non-Parametric P-values...")
    regions, regions_info_df, cluster_infos, sigThresh = perm_test_Perform_MikeXCOHEN_new(
        matrix_tmap, perms_tmap, n_subs, alpha_cluster_forming, pVal, tail=1
    )
    
    # P < pVal
    sig_clusters = regions_info_df[regions_info_df['pval_sumstats'] < pVal]
    sig_sizes = sig_clusters['size'].tolist()
    print(f"  > Found {len(regions)} clusters total. {len(sig_clusters)} are significant (P < {pVal}).")
    if len(sig_sizes) > 0:
        print(f"  > Significant cluster sizes (points): {sig_sizes}")

    # dict_Perform_PermTest_ResInfos[f'{paraname2}_auc_ElasticNetStrict'] = (
    #     regions, regions_info_df.to_dict(), cluster_infos, sigThresh
    # )
    dict_Perform_PermTest_ResInfos[f'{paraname2}_auc_ElasticNet'] = (
        regions, regions_info_df.to_dict(), cluster_infos, sigThresh
    )
    
    T_end = time.time()
    print(f"  > Done in {(T_end - T_start):.2f}s.")

save_stats_path = perf_dir / 'dict_Perform_PermTest_ResInfos.npz'
np.savez_compressed(save_stats_path, arr_0=dict_Perform_PermTest_ResInfos)
print(f"\nAll stats saved to {save_stats_path.name}")

Running Cluster-Based Permutation Statistics...

--- [Stats] Analyzing: Voluntary_Auditory_Session2 ---
  > Calculating Permutation T-maps for 20000 permutations...
  > Extracting Clusters & Calculating Non-Parametric P-values...
  > Found 66 clusters total. 1 are significant (P < 0.05).
  > Significant cluster sizes (points): [264]
  > Done in 54.54s.

All stats saved to dict_Perform_PermTest_ResInfos.npz


In [59]:
print("Have Completed Cluster-Based Permutation Statistics, and Showing the results ...")

pVal = 0.05
chance = 0.5

# # Load saved data
# aggregated_auc_path = perf_dir / 'Aggregated_AUC_CV.npz'
# dict_auc_results = np.load(aggregated_auc_path, allow_pickle=True)
# permutated_stats_path = perf_dir / 'dict_Perform_PermTest_ResInfos.npz'
# dict_Perform_PermTest_ResInfos = np.load(permutated_stats_path,allow_pickle=True)['arr_0'].item()
aggregated_auc_path = perf_dir / 'dict_auc_pre_cv_models.npz'
dict_auc_results = np.load(aggregated_auc_path, allow_pickle=True)['arr_0'].item()['ElasticNetStrict']
permutated_stats_path = perf_dir / 'dict_Perform_PermTest_ResInfos_p5.npz'
dict_Perform_PermTest_ResInfos = np.load(permutated_stats_path,allow_pickle=True)['arr_0'].item()

# for decis, stim, day in product(decis_names, stim_names, day_names):
#     decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
#     stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
#     paraname2 = f'{decision}_{stimulus}_{day}'
#     trialinfo = f'{decis}_{day}_No{stim}{period}'
for decis, stim, sess in product(decis_names[:], stim_names[:], sess_names[:]):
    decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
    stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
    session = 'Session1' if sess == 'Sess1' else 'Session2'
    paraname2 = f'{decision}_{stimulus}_{session}'
    trialinfo = f'{decis}_{sess}_No{stim}{period}'

    if paraname2 not in dict_auc_results:
        continue
    
    print(f"\n--- [Stats] {paraname2} ---")
    T_start = time.time()

    # Load subject data
    # subject_auc_dict = dict_auc_results[paraname2].item()
    subject_auc_dict = dict_auc_results[paraname2]
    _, null_distribution, sigThresh = dict_Perform_PermTest_ResInfos[f'{paraname2}_auc_ElasticNetStrict']
    
    # Calculate mean of n_repeat (20 times) for each subject, and stacked as (n_subs, n_times_train, n_times_general)
    matrix_auc_group = np.stack([auc_mat.mean(axis=0) for auc_mat in subject_auc_dict.values()])
    matrix_auc_mean = matrix_auc_group.mean(axis=0)
    print(f"  > MAX of AUC matrix {matrix_auc_mean.max():.3f}")
    n_subs = matrix_auc_group.shape[0]
    
    # Perform a t-test and the output shape (n_times_train, n_times_general).
    matrix_tmap, _ = ttest_1samp(matrix_auc_group, chance)

    # get significant clusters and their infos
    labeled = label(matrix_tmap > sigThresh,)
    regions = regionprops(labeled)
    regions_sumstat = np.array([np.sum([matrix_tmap[i,j] for (i,j) in r.coords]) for r in regions])
    cluster_thresh = np.percentile(null_distribution[:,1],(1-pVal)*100)
    len_sig_clusters = sum(regions_sumstat > cluster_thresh)
    print(f"  > Found {len(regions)} clusters total, {len_sig_clusters} are significant (P < {pVal}) ")
    sig_coords = regions[np.argmax(regions_sumstat)].coords
    p_cluster = (np.sum(np.abs(null_distribution[:,1]) > np.max(regions_sumstat)) + 1) / (n_repeat * n_perm + 1)
    print(f"  > Pcluster for the significant cluster: ", p_cluster)
    print(f"  > Training time range (ms): ", times_ms[[sig_coords[:,0].min(),sig_coords[:,0].max()-1]])
    print(f"  > Generalization time range (ms): ", times_ms[[sig_coords[:,1].min(),sig_coords[:,1].max()-1]])
    

Have Completed Cluster-Based Permutation Statistics, and Showing the results ...

--- [Stats] Cued_Visual_Session1 ---
  > MAX of AUC matrix 0.608
  > Found 120 clusters total, 1 are significant (P < 0.05) 
  > Pcluster for the significant cluster:  4.999750012499375e-05
  > Training time range (ms):  [-500    0]
  > Generalization time range (ms):  [-500    0]

--- [Stats] Cued_Visual_Session2 ---
  > MAX of AUC matrix 0.698
  > Found 120 clusters total, 2 are significant (P < 0.05) 
  > Pcluster for the significant cluster:  4.999750012499375e-05
  > Training time range (ms):  [-560    0]
  > Generalization time range (ms):  [-560    0]

--- [Stats] Cued_Auditory_Session1 ---
  > MAX of AUC matrix 0.566
  > Found 94 clusters total, 1 are significant (P < 0.05) 
  > Pcluster for the significant cluster:  4.999750012499375e-05
  > Training time range (ms):  [-370    0]
  > Generalization time range (ms):  [-390    0]

--- [Stats] Cued_Auditory_Session2 ---
  > MAX of AUC matrix 0.669
 

## Predictive Power Indix Calculation (PPI)

In [196]:
def compute_ch_contribs(D, fraction, thres=None):
    """
    Computes the normalized channel contributions based on frequency and magnitude.
    
    Parameters:
    D: ndarray of shape (n_subjects, n_repeat, n_splits, n_times, n_channels)
       Contains the absolute classifier coefficients.
    fraction: int or float, the total number of CV iterations (n_splits * n_repeat).
    thres: float, minimum absolute coefficient threshold to be considered active.
           If None, defaults to the minimum non-zero coefficient in D.
           
    Returns:
    all_c_norm: ndarray of shape (n_subjects, n_times, n_channels)
                Normalized channel contributions.
    """
    # Flatten the CV dimensions (repeats and splits) into a single axis
    # New shape: (n_subjects, n_cv_iterations, n_times, n_channels)
    n_subs, n_rep, n_spl, n_times, n_chans = D.shape
    D_reshaped = D.reshape(n_subs, n_rep * n_spl, n_times, n_chans)
    
    if thres is None:
        # Find the absolute minimum non-zero value across the entire tensor
        positive_vals = D_reshaped[D_reshaped > 0]
        thres = positive_vals.min() if len(positive_vals) > 0 else 0
        
    # 1. Calculate Frequency (f): How often a channel exceeds the threshold
    # all_f shape: (n_subs, n_times, n_chans)
    all_f = np.sum(D_reshaped > thres, axis=1) / fraction
    
    # 2. Calculate Magnitude (u): Average magnitude only when exceeding the threshold
    # all_u shape: (n_subs, n_times, n_chans)
    sum_D = np.sum(D_reshaped, axis=1)
    count_D = np.sum(D_reshaped > thres, axis=1) + 1e-10 # Prevent division by zero
    all_u = sum_D / count_D
    
    # 3. Calculate Contribution (c): Frequency * Magnitude
    all_c = all_f * all_u
    
    # 4. Normalize contributions across channels for each timepoint
    sum_c = all_c.sum(axis=-1, keepdims=True)
    sum_c[sum_c == 0] = 1e-10 # Prevent division by zero
    all_c_norm = all_c / sum_c
    
    return all_c_norm

def compute_ppi_log_norm(perf_matrix, coefs_matrix, fraction, chance_level=0.5, stat_time_range=None):
    """
    Computes the Log-normalized Predictive Performance Index (PPI).
    
    Parameters:
    perf_matrix: ndarray of shape (n_subs, n_times) - Diagonal elements of AUC/ACC matrices.
    coefs_matrix: ndarray of shape (n_subs, n_repeat, n_splits, n_times, n_chans) - Absolute classifier weights.
    fraction: int - Total number of CV iterations (n_splits * n_repeat).
    chance_level: float - 0.5 for AUC or balanced binary ACC.
    stat_time_range: slice or array - Specific time indices to retain for statistics.
    """

    # 1. Model-level PPI Calculation
    model_tps_ppi = (perf_matrix / chance_level) - 1
    # Rectify to zero: Any performance below chance contributes 0
    model_tps_ppi = np.where(model_tps_ppi > 0, model_tps_ppi, 0)
    
    # 2. Channel Contribution
    ch_contrib_norm = compute_ch_contribs(coefs_matrix, fraction, thres=None)
    
    # 3. Channel-level PPI Calculation
    # Broadcast model PPI to match channel dimensions: (n_subs, n_times, 1)
    ch_tps_ppi = ch_contrib_norm * model_tps_ppi[..., np.newaxis]
    # Transpose to conventional spatial-temporal layout: (n_subs, n_chans, n_times)
    ch_tps_ppi = np.transpose(ch_tps_ppi, (0, 2, 1))
    
    # Restrict to the predefined statistical time window
    if stat_time_range is None:
        stat_time_range = slice(None)
    ppi_subset = ch_tps_ppi[..., stat_time_range]
    
    # 4. Log-transformed Normalization
    # Note: RuntimeWarnings for divide by zero (log(0)) are expected here and handled below
    with np.errstate(divide='ignore'):
        ppi_indi_log = np.log(ppi_subset)
    
    # Handle -inf values resulting from log(0)
    flag_indi_isinf = np.isinf(ppi_indi_log)
    if flag_indi_isinf.any():
        valid_log_vals = ppi_indi_log[~flag_indi_isinf]
        # Define limits based on the lower tail of the valid empirical distribution
        limit_min, limit_median, limit_max = np.quantile(valid_log_vals, [0.001, 0.01, 0.04])
        
        # Generate random noise matching the extreme lower tail parameters
        mu = limit_max
        sigma = (limit_max - limit_min) / 4
        ppi_indi_log[flag_indi_isinf] = np.random.normal(mu, sigma, flag_indi_isinf.sum())
        
    # 5. Grand Standardization & Clipping
    mu_indi_grand = ppi_indi_log.mean()
    std_indi_grand = ppi_indi_log.mean(axis=0).std()
    # Calculate Z-score based on grand parameters
    ppi_indi_log_norm1 = (ppi_indi_log - mu_indi_grand) / std_indi_grand
    # Clip extreme lower outliers (bottom 1%) to stabilize topographical variance
    clip_lower_bound = np.quantile(ppi_indi_log_norm1, 0.01)
    ppi_indi_log_norm1 = np.clip(ppi_indi_log_norm1, clip_lower_bound, ppi_indi_log_norm1.max())
    
    return ppi_indi_log_norm1

In [203]:
# Dictionary to store the final computed PPI matrices

index_metric = 'auc'  # Choose 'auc' or 'acc'
chance = 0.5
fraction_val = n_splits * n_repeat  # Typically 8 * 20 = 160
stat_time_range_idx = slice(0, 96) # id_time_zero = 95
print(f"Calculating Log-normalized Channel PPI based on {index_metric.upper()}...")

# Load saved data
aggregated_auc_path = perf_dir / f'Aggregated_{index_metric.upper()}_CV.npz'
dict_auc_results = np.load(aggregated_auc_path, allow_pickle=True)

dict_ppi_results = {}

# for decis, stim, day in product(decis_names, stim_names, day_names):
#     decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
#     stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
#     paraname2 = f'{decision}_{stimulus}_{day}'
#     trialinfo = f'{decis}_{day}_No{stim}{period}'
for decis, stim, sess in product(decis_names[1:], stim_names[1:], sess_names[1:]):
    decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
    stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
    session = 'Session1' if sess == 'Sess1' else 'Session2'
    paraname2 = f'{decision}_{stimulus}_{session}'
    trialinfo = f'{decis}_{sess}_No{stim}{period}'

    if paraname2 not in dict_auc_results:
        continue
    
    print(f"\n--- [Stats] Calculating: {paraname2} ---")
    
    # 1. Load Aggregated Performance (AUC or ACC)
    subject_auc_dict = dict_auc_results[paraname2].item()
    auc_matrix = np.stack([d for d in subject_auc_dict.values()]).mean(axis=1)
    perf_diag = np.diagonal(auc_matrix, axis1=1, axis2=2)
    
    # 2. Load Aggregated Channel Coefficients
    # (n_subs, n_repeats, n_splits, n_times, n_chans)]
    all_ch_coefs = []
    for sid in subject_auc_dict:
        sid_folder = perf_dir / str(sid)
        savefile_template = f'{n_repeat}Rounds_{n_splits}Folds_ElasticNet_{trialinfo}_S{sid}'
        # savefile_template = f'{n_repeat}Rounds_{n_splits}Folds_ElasticNetStrict_{trialinfo}_S{sid}'
        try:
            ch_coefs_cv = np.load(sid_folder / f'{savefile_template}_ch_coefs_matrix_cv.npz')['arr_0']
            all_ch_coefs.append(np.abs(ch_coefs_cv))
        except FileNotFoundError:
            print(f"  [Warning] Missing channel coefficent file for S{sid}. Statistics will be compromised!")
            continue
    all_ch_coefs = np.asanyarray(all_ch_coefs)
    
    # 3. Compute final PPI using the custom pipeline
    final_ppi_norm = compute_ppi_log_norm(
        perf_matrix=perf_diag, 
        coefs_matrix=np.abs(all_ch_coefs), # Ensure inputs are absolute values
        fraction=fraction_val,
        chance_level=chance, 
        stat_time_range=stat_time_range_idx
    )
    
    dict_ppi_results[paraname2] = final_ppi_norm

np.savez_compressed(perf_dir / 'dict_PPI_Log_Norm.npz', **dict_ppi_results)
print("\nPPI Calculation and Normalization complete.")

Calculating Log-normalized Channel PPI based on AUC...

--- [Stats] Calculating: Voluntary_Auditory_Session2 ---

PPI Calculation and Normalization complete.


### Statistics of Log-normalized PPIs

In [2]:
print(f"Statistics of Log-normalized PPIs based on Predefined ROIs and time windows ...")

# Labeling based on your 0.05/0.1 threshold
def tag_significance(row):
    p_unc = row['p_unc']
    p_fdr = row['p_fdr']
    if p_fdr < 0.05: return 'Sign.'
    if p_unc < 0.05 and p_fdr < 0.10: return 'Margin. Sign.'
    return 'Non-Sign.'

## Predefined ROIs and time windows
roi_configs = {
    'Session1': {
        'Occ': ['O1', 'Oz', 'O2'], 'Temp': ['T7', 'T8', 'TP7', 'TP8'], 'times': [-100, 0]
    },
    'Session2': {
        'Visual':   {'Occ': ['O1', 'Oz', 'O2'], 'Temp': ['T7', 'T8', 'TP7', 'TP8'], 'times': [-100, 0]},
        'Cued_Auditory_1': {'Occ': ['O1', 'Oz', 'O2'], 'Temp': ['T7', 'T8', 'TP7', 'TP8'], 'times': [-150, -50]},
        'Cued_Auditory_2': {'Occ': ['O1', 'Oz', 'O2'], 'Temp': ['FC5', 'FT7'], 'times': [-50, 0]},
        'Voluntart_Auditory': {'Occ': ['O1', 'Oz', 'O2'], 'Temp': ['T7', 'T8', 'TP7', 'TP8'], 'times': [-150, 0]},
    }
}

# Load saved data
# dict_PPI_Log_Norm = np.load( perf_dir/'dict_PPI_Log_Norm.npz',allow_pickle=True)
dict_PPI_Log_Norm = np.load( perf_dir/'dict_cluster_Indi_PPI_Log_Norm.npy',allow_pickle=True)
dict_PPI_Log_Norm = dict_PPI_Log_Norm.item()

for decis in decis_names[:]:
    decision = 'Cued' if decis.startswith('Cue') else 'Voluntary'
    print(f"\n{'='*50}\n [STATISTICS FOR PPI] | DECISION: {decision.upper()}\n{'='*50}")

    roi_stat_accumulator = []
    # for stim, day in product(stim_names, day_names):
    #     stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
    #     paraname2 = f'{decision}_{stimulus}_{day}'
    #     trialinfo = f'{decis}_{day}_No{stim}{period}'
    #     ppi_indi_log_norm = dict_PPI_Log_Norm[paraname2]
    for stim, sess in product(stim_names[:], sess_names[:]):
        stimulus = 'Visual' if stim.startswith('Vis') else 'Auditory'
        session = 'Session1' if sess == 'Sess1' else 'Session2'
        paraname2 = f'{decision}_{stimulus}_{session}'
        trialinfo = f'{decis}_{sess}_No{stim}{period}'
        saveinfo = f'Averaged_{n_repeat}Rounds_{n_splits}Folds_PPI_AUC_Contribs_ElasticNetStrict_{trialinfo}'
        ppi_indi_log_norm = dict_PPI_Log_Norm[saveinfo]['ppi_indi_log_norm1']

        applicable_rois = []
        if session == 'Session1':
            applicable_rois.append(roi_configs['Session1'])
        else:
            if stimulus == 'Visual':
                applicable_rois.append(roi_configs['Session2']['Visual'])
            else:
                if decision == 'Cued':
                    applicable_rois.append(roi_configs['Session2']['Cued_Auditory_1'])
                    applicable_rois.append(roi_configs['Session2']['Cued_Auditory_2'])
                else:
                    applicable_rois.append(roi_configs['Session2']['Voluntart_Auditory'])
        
        for roi_info in applicable_rois:
            # print(paraname2,roi_info)
            # Electrodes: # 1. Occipital 2. Temporal 3. Remaining (Rest)
            target_ch_occ  = np.where(np.isin(ch_names, roi_info['Occ']))[0]
            target_ch_temp = np.where(np.isin(ch_names, roi_info['Temp']))[0]
            all_ch_idx = np.arange(len(ch_names))
    
            # Spatial and temporal mask for data
            mask_occ = np.isin(all_ch_idx, target_ch_occ)
            mask_temp = np.isin(all_ch_idx, target_ch_temp)
            mask_rest = ~mask_occ & ~mask_temp
            t_mask = (times_ms_pre >= roi_info['times'][0]) & (times_ms_pre <= roi_info['times'][1])
    
            # Averaged PPI
            group_means =dict(
                Occ = ppi_indi_log_norm[:, mask_occ, :][:, :, t_mask].mean(axis=(1, 2)),
                Temp = ppi_indi_log_norm[:, mask_temp, :][:, :, t_mask].mean(axis=(1, 2)),
                Rest = ppi_indi_log_norm[:, mask_rest, :][:, :, t_mask].mean(axis=(1, 2))
            )
    
            # Statistical Testing
            comparisons = [('Occ', 'Rest'), ('Temp', 'Rest')]
            for name1, name2 in comparisons:
                test_res = pg.wilcoxon(group_means[name1], group_means[name2], alternative='two-sided', method="exact")
                t_val, p_unc, method = test_res['W-val'].values[0], test_res['p-val'].values[0], 'Wilcoxon'
                h_g = pg.compute_effsize(group_means[name1], group_means[name2], paired=True, eftype='hedges')
                
                roi_stat_accumulator.append({
                    'Type': paraname2, 'Comparison': f"{name1} vs {name2}", 'Times': roi_info['times'],
                    'Method': method, 'Stat': t_val, 'p_unc': p_unc, 'Hedges_g': h_g
                })

    # Apply independent FDR calibration
    if roi_stat_accumulator:
        df_roi_results = pd.DataFrame(roi_stat_accumulator)
        h_reject, p_fdr = pg.multicomp(df_roi_results['p_unc'].values, method='fdr_bh')
        df_roi_results['p_fdr'] = p_fdr
        df_roi_results['Significance'] = df_roi_results.apply(tag_significance, axis=1)
        
        print(df_roi_results.to_string(index=False, formatters={
            'Stat': '{:0.3f}'.format, 'Shap_p': '{:0.4f}'.format, 'Lev_p': '{:0.4f}'.format, 'Bart_p': '{:0.4f}'.format,
            'p_unc': '{:0.4f}'.format, 'p_fdr': '{:0.4f}'.format, 'Hedges_g': '{:0.3f}'.format
        }))
    

Statistics of Log-normalized PPIs based on Predefined ROIs and time windows ...

 [STATISTICS FOR PPI] | DECISION: CUED
                  Type   Comparison       Times   Method   Stat  p_unc Hedges_g  p_fdr  Significance
  Cued_Visual_Session1  Occ vs Rest   [-100, 0] Wilcoxon  0.000 0.0000    0.996 0.0003         Sign.
  Cued_Visual_Session1 Temp vs Rest   [-100, 0] Wilcoxon 33.000 0.0739    0.480 0.0924     Non-Sign.
  Cued_Visual_Session2  Occ vs Rest   [-100, 0] Wilcoxon 29.000 0.0443    0.771 0.0739 Margin. Sign.
  Cued_Visual_Session2 Temp vs Rest   [-100, 0] Wilcoxon 27.000 0.0335    0.593 0.0671 Margin. Sign.
Cued_Auditory_Session1  Occ vs Rest   [-100, 0] Wilcoxon 38.000 0.0714    0.348 0.0924     Non-Sign.
Cued_Auditory_Session1 Temp vs Rest   [-100, 0] Wilcoxon 11.000 0.0008    0.527 0.0042         Sign.
Cued_Auditory_Session2  Occ vs Rest [-150, -50] Wilcoxon 40.000 0.0887    0.243 0.0985     Non-Sign.
Cued_Auditory_Session2 Temp vs Rest [-150, -50] Wilcoxon 13.000 0.0013  